In [78]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from utils import *

In [79]:
results = {"CV": np.array([]),
           "ACC": np.array([]),
           "PRE": np.array([]),
           "REC": np.array([]),
           "F1": np.array([])}

comments, sentiments = load_data("dataset/reviews.csv")

# create dictionary mapping from emoji to sentiment
emoji_dict = create_emoji_dict(comments)
for i in range(len(comments)):
    comments[i] = normalize_emoji(comments[i], emoji_dict)
    comments[i] = clean_text(comments[i])
    comments[i] = remove_stopwords(comments[i])

# remove null comments
mask = comments != ""
comments = comments[mask]
sentiments = sentiments[mask]


permutations = np.random.permutation(len(comments))
comments = comments[permutations]
sentiments = sentiments[permutations]

In [80]:


X_train, X_test, y_train, y_test = train_test_split(comments, sentiments, test_size=0.02)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l1")

X_train_vector = vectorizer.fit_transform(X_train)
X_test_vector = vectorizer.transform(X_test)

In [82]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train_vector, y_train)
y_pred = model.predict(X_test_vector)

accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average="weighted")
precision = precision_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print(f'accuracy: {accuracy:.3f}')
print(f'recall: {recall:.3f}')
print(f'precision: {precision:.3f}')
print(f'f1: {f1:.3f}')

accuracy: 0.676
recall: 0.676
precision: 0.690
f1: 0.677


In [776]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

model = LogisticRegression()

param_grid = {
    "C": [0.1, 0.01, 0.001],
    "solver": ["lbfgs", "sag", "saga"],
    "verbose": [0, 1],
    "l1_ratio": [0],
    "class_weight": ["balanced", None],
    "fit_intercept": [True, False],
}

clf = GridSearchCV(model, param_grid, cv=5, n_jobs=-1, verbose=1, scoring="accuracy")
clf.fit(X_train_vector, y_train)

print("Best params: ", clf.best_params_)
print("Best CV score:", clf.best_score_)
print("Test score:  ", clf.best_estimator_.score(X_test_vector, y_test))

Fitting 5 folds for each of 72 candidates, totalling 360 fits


KeyboardInterrupt: 

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed

param_grid = [

    {
        "clf": [SGDClassifier(max_iter=1000)],
        "clf__loss":         ["hinge", "log_loss",],
        "clf__alpha":        [0.0001, 0.001, 0.01],
        "clf__class_weight": ["balanced", None],
        "clf__shuffle":      [True, False]
    },
]

clf = GridSearchCV(
    Pipeline([("clf", LogisticRegression())]),  # placeholder
    param_grid,
    cv=5,
    n_jobs=1,
    verbose=1,
    scoring="accuracy",
)
clf.fit(X_train_vector, y_train)

print("Best params: ", clf.best_params_)
print("Best CV score:", clf.best_score_)
print("Test score:   ", clf.best_estimator_.score(X_test_vector, y_test))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best params:  {'clf': SGDClassifier(), 'clf__alpha': 0.0001, 'clf__class_weight': 'balanced', 'clf__loss': 'hinge', 'clf__shuffle': False}
Best CV score: 0.723828125
Test score:    0.6857142857142857


In [797]:
C = []
for i in range(1,10):
    C.append(10.5 + i/10)
print(C)

[10.6, 10.7, 10.8, 10.9, 11.0, 11.1, 11.2, 11.3, 11.4]


In [ ]:
from sklearn.linear_model import SGDClassifier
results = np.array([])

# model = SGDClassifier(
#     loss="log_loss", 
#     class_weight="balanced", 
#     shuffle=True, 
#     max_iter=100, 
#     penalty="l2", 
#     alpha=0.0000001, 
#     eta0=0.1
# )


for i in range(100):

    model = SGDClassifier(
        loss="log_loss", 
        class_weight="balanced", 
        shuffle=True, 
        max_iter=1000, 
        penalty=None, 
        learning_rate="optimal", 
        alpha=0.0001, 
        eta0=1, 
        n_jobs=20
        )
    
    # model = LogisticRegression(
    #     class_weight="balanced",
    #     l1_ratio=1,
    #     solver="saga",
    #     max_iter=5000,
    #     tol=0.1,
    #     n_jobs=-1
    #     )

    model.fit(X_train_vector, y_train)
    y_pred = model.predict(X_test_vector)

    acc = accuracy_score(y_test, y_pred)
    results = np.append(results, acc)

In [760]:
print(f'Max acc: {results.max()}')
print(f'Miin acc: {results.min()}')
print(f'Average acc: {results.mean()}')
print(f'model acc: {acc}')

Max acc: 0.7333333333333333
Miin acc: 0.7047619047619048
Average acc: 0.7134285714285714
model acc: 0.7047619047619048
